# Adaptive Bangla CAPTCHA — RL Algorithms Walkthrough

End-to-end walkthrough of the Reinforcement Learning pipeline for adaptive CAPTCHA difficulty selection.

**Algorithms covered:** PPO, Soft-PPO, Dual Generator (DG)

**Pipeline:**
1. Environment & state representation
2. Simulated human/bot telemetry data
3. Training PPO / Soft-PPO / DG agents
4. Evaluation & figure generation

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['savefig.dpi'] = 150

# Project paths
PROJECT_ROOT = os.path.join(os.getcwd(), '..', 'Adaptive-Bangla-CAPTCHA')
RL_ROOT = os.path.join(PROJECT_ROOT, 'rl_captcha')
if RL_ROOT not in sys.path:
    sys.path.insert(0, RL_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'RL root: {RL_ROOT}')

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Security Actions & Environment

In [ ]:
from environment.security_actions import (
    SECURITY_ACTIONS, ACTION_LABELS, NUM_ACTIONS,
    action_to_difficulty, action_name,
)

print(f'Number of actions: {NUM_ACTIONS}')
print()
for a, label in ACTION_LABELS.items():
    diff = action_to_difficulty(a)
    print(f'  Action {a}: {label}  (difficulty={diff})')

In [ ]:
from environment.captcha_env import CaptchaEnv
from environment.state import StateBuilder
from environment.reward import RewardFunction

STATE_DIM = 20
ACTION_DIM = NUM_ACTIONS  # 7

env = CaptchaEnv(state_dim=STATE_DIM, num_actions=ACTION_DIM)
state_builder = StateBuilder(feature_dim=STATE_DIM)
reward_fn = RewardFunction()

print(f'State dimension: {STATE_DIM}')
print(f'Action dimension: {ACTION_DIM}')
print(f'\nState features:')
feature_keys = [
    'mouse_avg_speed', 'mouse_std_speed', 'mouse_path_length',
    'mouse_idle_periods', 'mouse_click_count', 'mouse_efficiency',
    'kb_dwell_mean', 'kb_dwell_std', 'kb_flight_mean',
    'kb_speed_cpm', 'kb_rhythm', 'kb_correction_ratio',
    'touch_count', 'touch_jitter', 'touch_force_mean',
    'scroll_events', 'scroll_speed', 'scroll_smoothness',
    'bot_score', 'confidence',
]
for i, k in enumerate(feature_keys):
    print(f'  [{i:2d}] {k}')

## 3. Simulated Telemetry Data

We generate realistic synthetic mouse/keyboard events to simulate human and bot sessions.

In [ ]:
def simulate_human_session():
    """Generate realistic human behavioral telemetry."""
    n_mouse = np.random.randint(30, 120)
    mouse_events = []
    base_t = 0.0
    for i in range(n_mouse):
        dt = np.random.exponential(30)
        base_t += dt
        speed = max(0, np.random.normal(500, 200))
        mouse_events.append({
            'x': float(np.random.uniform(100, 1800)),
            'y': float(np.random.uniform(100, 900)),
            'timestamp': base_t,
            'button': 0,
            'click_type': 'click' if np.random.random() < 0.15 else '',
            'speed': float(speed),
        })

    n_keys = np.random.randint(15, 50)
    keyboard_events = []
    base_t = 0.0
    for i in range(n_keys):
        dt = np.random.exponential(80)
        base_t += dt
        key = chr(np.random.randint(97, 123))
        keyboard_events.append({
            'type': 'keydown', 'key': key, 'code': f'Key{key.upper()}',
            'timestamp': base_t,
        })
        keyboard_events.append({
            'type': 'keyup', 'key': key, 'code': f'Key{key.upper()}',
            'timestamp': base_t + np.random.normal(60, 20),
        })

    return {'is_bot': False, 'mouse_events': mouse_events, 'keyboard_events': keyboard_events}


def simulate_bot_session():
    """Generate bot-like behavioral telemetry."""
    n_mouse = np.random.randint(5, 15)
    mouse_events = []
    for i in range(n_mouse):
        mouse_events.append({
            'x': float(np.random.uniform(0, 1920)),
            'y': float(np.random.uniform(0, 1080)),
            'timestamp': float(i * np.random.uniform(1, 5)),
            'button': 0,
            'click_type': '',
            'speed': float(np.random.uniform(0, 100)),
        })

    n_keys = np.random.randint(3, 10)
    keyboard_events = []
    for i in range(n_keys):
        t = float(i * np.random.uniform(20, 80))
        key = chr(np.random.randint(97, 123))
        keyboard_events.append({
            'type': 'keydown', 'key': key, 'code': f'Key{key.upper()}',
            'timestamp': t,
        })
        keyboard_events.append({
            'type': 'keyup', 'key': key, 'code': f'Key{key.upper()}',
            'timestamp': t + np.random.uniform(10, 50),
        })

    return {'is_bot': True, 'mouse_events': mouse_events, 'keyboard_events': keyboard_events}


# Test
human = simulate_human_session()
bot = simulate_bot_session()
print(f'Human session: {len(human["mouse_events"])} mouse, {len(human["keyboard_events"])} keyboard events')
print(f'Bot session:   {len(bot["mouse_events"])} mouse, {len(bot["keyboard_events"])} keyboard events')

## 4. LSTM Network Architecture

In [ ]:
from agent.lstm_network import LSTMActorCritic, LSTMNetwork

# Visualize the architecture
net = LSTMActorCritic(obs_dim=STATE_DIM, action_dim=ACTION_DIM, hidden_dim=128, lstm_dim=64)
print('LSTMActorCritic Architecture:')
print(net)

total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 5. Training Loop

We train all three agents (PPO, Soft-PPO, DG) for comparison. Each uses:
- 20-dimensional state from behavioral features
- 7 security actions
- LSTM-based actor-critic
- Simulated bot ratio of 30%

In [ ]:
from agent.ppo import PPOAgent
from agent.softppo import SoftPPOAgent
from agent.dg import DualGeneratorAgent

NUM_EPISODES = 500
BOT_RATIO = 0.3

def simulate_outcome(difficulty, is_bot):
    if is_bot:
        correct_prob = max(0.1, 0.9 - difficulty * 0.25)
    else:
        correct_prob = max(0.3, 0.95 - difficulty * 0.1)
    is_correct = np.random.random() < correct_prob
    solve_time = np.random.exponential(3000 + difficulty * 1000)
    return is_correct, solve_time


def train_agent(agent, agent_name, num_episodes=NUM_EPISODES, bot_ratio=BOT_RATIO):
    env = CaptchaEnv(state_dim=STATE_DIM, num_actions=ACTION_DIM)
    rewards_log = []
    difficulty_log = []
    human_accs = []
    bot_accs = []
    total_h_correct, total_h = 0, 0
    total_b_correct, total_b = 0, 0

    start = time.time()
    for ep in range(num_episodes):
        is_bot = np.random.random() < bot_ratio
        session = simulate_bot_session() if is_bot else simulate_human_session()

        state = env.reset(mouse_events=session['mouse_events'], keyboard_events=session['keyboard_events'])
        ep_reward = 0.0
        diff_chosen = 1
        done = False

        while not done:
            action, log_prob, value = agent.select_action(state)
            difficulty = int(np.clip(action, 0, ACTION_DIM - 1)) + 1
            diff_chosen = difficulty
            next_state, _, done, info = env.step(action)
            is_correct, solve_time = simulate_outcome(difficulty, is_bot)
            reward = env.receive_outcome(is_correct=is_correct, is_bot=is_bot, solve_time_ms=solve_time)
            agent.store_transition(state, action, log_prob, reward, done, value)
            state = next_state
            ep_reward += reward

            if is_bot:
                total_b += 1
                if is_correct: total_b_correct += 1
            else:
                total_h += 1
                if is_correct: total_h_correct += 1

        agent.update()
        rewards_log.append(ep_reward)
        difficulty_log.append(diff_chosen)
        human_accs.append(total_h_correct / max(total_h, 1))
        bot_accs.append(total_b_correct / max(total_b, 1))

        if (ep + 1) % 100 == 0:
            avg_r = np.mean(rewards_log[-100:])
            print(f'  [{agent_name}] Ep {ep+1}/{num_episodes} | Avg Reward: {avg_r:.3f} | Diff: {np.mean(difficulty_log[-100:]):.1f}')

    elapsed = time.time() - start
    print(f'  [{agent_name}] Done in {elapsed:.1f}s | Final avg reward: {np.mean(rewards_log[-50:]):.3f}')

    return {
        'rewards': rewards_log,
        'difficulties': difficulty_log,
        'human_accs': human_accs,
        'bot_accs': bot_accs,
        'time': elapsed,
    }

In [ ]:
# Train PPO
print('=' * 60)
print('Training PPO Agent')
print('=' * 60)
ppo_agent = PPOAgent(obs_dim=STATE_DIM, action_dim=ACTION_DIM, device=device)
ppo_results = train_agent(ppo_agent, 'PPO')

In [ ]:
# Train Soft-PPO
print('=' * 60)
print('Training Soft-PPO Agent')
print('=' * 60)
softppo_agent = SoftPPOAgent(obs_dim=STATE_DIM, action_dim=ACTION_DIM, device=device)
softppo_results = train_agent(softppo_agent, 'Soft-PPO')

In [ ]:
# Train Dual Generator (DG)
print('=' * 60)
print('Training Dual Generator Agent')
print('=' * 60)
dg_agent = DualGeneratorAgent(obs_dim=STATE_DIM, action_dim=ACTION_DIM, device=device)
dg_results = train_agent(dg_agent, 'DG')

## 6. Evaluation & Comparison

In [ ]:
def smooth(values, window=20):
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window)/window, mode='valid')


fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RL Agent Comparison — Adaptive CAPTCHA Difficulty', fontsize=14, fontweight='bold')

results = {
    'PPO': ppo_results,
    'Soft-PPO': softppo_results,
    'DG': dg_results,
}
colors = {'PPO': '#2196F3', 'Soft-PPO': '#FF9800', 'DG': '#4CAF50'}

# Plot 1: Episode Rewards
ax = axes[0, 0]
for name, res in results.items():
    ax.plot(smooth(res['rewards']), label=name, color=colors[name], linewidth=1.5)
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.set_title('Episode Reward (smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Difficulty Chosen
ax = axes[0, 1]
for name, res in results.items():
    ax.plot(smooth(res['difficulties']), label=name, color=colors[name], linewidth=1.5)
ax.set_xlabel('Episode')
ax.set_ylabel('Difficulty')
ax.set_title('Average Difficulty Selected (smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Human Accuracy
ax = axes[1, 0]
for name, res in results.items():
    ax.plot(smooth(res['human_accs']), label=name, color=colors[name], linewidth=1.5)
ax.set_xlabel('Episode')
ax.set_ylabel('Accuracy')
ax.set_title('Human Pass Rate (smoothed)')
ax.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5, label='Target 95%')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Bot Accuracy
ax = axes[1, 1]
for name, res in results.items():
    ax.plot(smooth(res['bot_accs']), label=name, color=colors[name], linewidth=1.5)
ax.set_xlabel('Episode')
ax.set_ylabel('Accuracy')
ax.set_title('Bot Pass Rate (smoothed) — Lower is Better')
ax.axhline(y=0.3, color='gray', linestyle='--', alpha=0.5, label='Target <30%')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rl_agent_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: rl_agent_comparison.png')

In [ ]:
# Summary table
print(f'{"Agent":<12} {"Avg Reward":>12} {"Avg Difficulty":>16} {"Human Acc":>12} {"Bot Acc":>12} {"Time (s)":>10}')
print('-' * 76)
for name, res in results.items():
    last50 = slice(-50, None) if len(res['rewards']) >= 50 else slice(None)
    print(
        f'{name:<12} '
        f'{np.mean(res["rewards"][last50]):>12.4f} '
        f'{np.mean(res["difficulties"][last50]):>16.2f} '
        f'{res["human_accs"][-1]:>12.4f} '
        f'{res["bot_accs"][-1]:>12.4f} '
        f'{res["time"]:>10.1f}'
    )

## 7. Save Checkpoints

In [ ]:
checkpoint_dir = os.path.join(RL_ROOT, 'agent', 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

ppo_agent.save(os.path.join(checkpoint_dir, 'ppo_best.pt'))
softppo_agent.save(os.path.join(checkpoint_dir, 'softppo_best.pt'))
dg_agent.save(os.path.join(checkpoint_dir, 'dg_best.pt'))

print(f'Checkpoints saved to: {checkpoint_dir}')
print(os.listdir(checkpoint_dir))

## 8. Inference Demo

Run a trained agent on a single session to see the decision pipeline.

In [ ]:
from inference.predict import CaptchaPredictor

# Load PPO predictor
predictor = CaptchaPredictor(
    agent_type='ppo',
    model_path=os.path.join(checkpoint_dir, 'ppo_best.pt'),
)

# Simulate a suspicious session
suspicious = simulate_bot_session()
result = predictor.predict_action(
    mouse_events=suspicious['mouse_events'],
    keyboard_events=suspicious['keyboard_events'],
    previous_difficulty=1,
    attempt_count=0,
    bot_score=0.65,
    confidence=0.7,
)

print('Suspicious session prediction:')
print(f'  Action: {result["action"]} ({result["action_name"]})')
print(f'  Difficulty: {result["difficulty"]}')
print(f'  Decision: {result["decision"]}')
print()

# Simulate a trusted human session
trusted = simulate_human_session()
result2 = predictor.predict_action(
    mouse_events=trusted['mouse_events'],
    keyboard_events=trusted['keyboard_events'],
    previous_difficulty=1,
    attempt_count=0,
    bot_score=0.1,
    confidence=0.9,
)

print('Trusted session prediction:')
print(f'  Action: {result2["action"]} ({result2["action_name"]})')
print(f'  Difficulty: {result2["difficulty"]}')
print(f'  Decision: {result2["decision"]}')

## 9. Reward Function Analysis

Visualize the reward landscape for different action/bot combinations.

In [ ]:
reward_fn = RewardFunction()

action_names = ['Allow', 'Observe', 'Easy', 'Medium', 'Hard', 'Honeypot', 'Block']
scenarios = [
    {'label': 'Human correct', 'is_correct': True, 'is_bot': False, 'difficulty': 2},
    {'label': 'Human incorrect', 'is_correct': False, 'is_bot': False, 'difficulty': 2},
    {'label': 'Bot correct', 'is_correct': True, 'is_bot': True, 'difficulty': 2},
    {'label': 'Bot incorrect', 'is_correct': False, 'is_bot': True, 'difficulty': 2},
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)
fig.suptitle('Reward Function — Action vs Scenario', fontsize=13, fontweight='bold')

for ax, scenario in zip(axes, scenarios):
    rewards = []
    for action in range(7):
        r, _ = reward_fn.compute(
            is_correct=scenario['is_correct'],
            is_bot=scenario['is_bot'],
            difficulty=scenario['difficulty'],
            solve_time_ms=5000,
            attempt_count=1,
            action=action,
        )
        rewards.append(r)

    bar_colors = ['#f44336' if r < 0 else '#4CAF50' for r in rewards]
    ax.bar(action_names, rewards, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_title(scenario['label'])
    ax.set_ylabel('Reward' if scenario == scenarios[0] else '')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('reward_landscape.png', bbox_inches='tight')
plt.show()
print('Saved: reward_landscape.png')